In [6]:
import os
import glob
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# TEMPLATE LAYER

In [10]:
# read study area
study_area = gpd.read_file('layer/bbox_study_area.gpkg')

study_area_fi = study_area.to_crs(3067)

In [11]:
study_area_fi.crs

<Projected CRS: EPSG:3067>
Name: ETRS89 / TM35FIN(E,N)
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Finland - onshore and offshore.
- bounds: (19.08, 58.84, 31.59, 70.09)
Coordinate Operation:
- name: TM35FIN
- method: Transverse Mercator
Datum: European Terrestrial Reference System 1989 ensemble
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [12]:
import rasterio
from rasterio.transform import from_bounds
import numpy as np

# Bounding box from study_area GeoDataFrame
mbbox = study_area_fi.total_bounds.tolist()  # [minx, miny, maxx, maxy]

# Desired resolution in meters (10 m)
res = 10

# Calculate number of columns and rows
width = max(int((mbbox[2] - mbbox[0]) / res), 1)
height = max(int((mbbox[3] - mbbox[1]) / res), 1)

# Create transform
transform = from_bounds(*mbbox, width=width, height=height)

# Create raster filled with 999
template = np.full((height, width), 999, dtype="float32")

# Save template
out_meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": "EPSG:3067",  # Finnish CRS
    "transform": transform
}

with rasterio.open("Norm/template_999.tif", "w", **out_meta) as dst:
    dst.write(template, 1)

print("Template raster created:", template.shape)

Template raster created: (10520, 8194)


# TEMPLATE VARIABLE

In [13]:
# Read template raster
with rasterio.open("Norm/template_999.tif") as src:
    template = src.read(1)
    template_meta = src.meta.copy()
    template_shape = template.shape
    template_transform = src.transform
    template_crs = src.crs

print("Template shape:", template_shape)
print("Template CRS:", template_crs)

Template shape: (10520, 8194)
Template CRS: EPSG:3067


In [5]:
folder = "Norm/"
weighted_paths = sorted(glob.glob(os.path.join(folder, "*weight*.tif")))

print("Weighted rasters found:")
for p in weighted_paths:
    print(p)

Weighted rasters found:
Norm/Slope_weighted.tif
Norm/canopy_weighted.tif
Norm/dtw_weighted.tif
Norm/peatland_weighted.tif
Norm/roads_weighted.tif
Norm/superdeposits_weighted.tif
Norm/water_weighted.tif


In [17]:
for path in weighted_paths:
    
    with rasterio.open(path) as src:
        src_data = src.read(1)
        data_crs = src.crs

        print(path, data_crs)
        
        # Prepare empty array with template shape
        dst_data = np.full(template_shape, np.nan, dtype="float32")
        
        # Reproject / resample to template grid
        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=template_transform,
            dst_crs=template_crs,
            resampling=Resampling.nearest
        )
        
        # Create output filename
        base = os.path.basename(path).replace(".tif", "_template.tif")
        out_path = os.path.join("Template", base)
        
        # Update metadata to template grid
        out_meta = template_meta.copy()
        out_meta.update(dtype="float32", count=1)
        
        # Save
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(dst_data, 1)
        
        print("Saved:", out_path, '\n')

Norm/Slope_weighted.tif EPSG:3067
Saved: Template/Slope_weighted_template.tif 

Norm/canopy_weighted.tif EPSG:3067
Saved: Template/canopy_weighted_template.tif 

Norm/dtw_weighted.tif EPSG:3067
Saved: Template/dtw_weighted_template.tif 

Norm/peatland_weighted.tif EPSG:9391
Saved: Template/peatland_weighted_template.tif 

Norm/roads_weighted.tif EPSG:3067
Saved: Template/roads_weighted_template.tif 

Norm/superdeposits_weighted.tif EPSG:3067
Saved: Template/superdeposits_weighted_template.tif 

Norm/water_weighted.tif EPSG:3067
Saved: Template/water_weighted_template.tif 

